# Handwritten Text Recognition (HTR) with TrOCR

**Model:** `microsoft/trocr-base-handwritten`  
**Dataset:** IAM handwriting subset (4,899 line images)  
**Task:** Handwritten line-image → text transcription

> **Experiment note:** The current pipeline uses pseudo-labels generated by the pretrained TrOCR model because the local image dataset used in this experiment did not include transcription labels in the files consumed by the notebook. Its validation metrics should therefore not be interpreted as equivalent to evaluation against independently verified ground-truth transcriptions.

### Engineering features
- CUDA/GPU detection with CPU fallback
- FP16 mixed precision when CUDA is available
- Checkpointing and resume support
- Data augmentation for handwriting variation
- Cosine learning-rate schedule
- CER, WER, and exact-match evaluation
- Beam-search inference
- Optional dictionary-based spell correction




## 📁 Step 1: Setup & Imports


In [ ]:
%pip install pandas matplotlib tqdm


In [ ]:
%pip install transformers datasets torchvision pillow jiwer accelerate -q


In [ ]:
import os, sys, json, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    default_data_collator
)

warnings.filterwarnings('ignore')
print("All imports successful")


In [ ]:
import multiprocessing
multiprocessing.set_start_method("spawn", force=True)


In [ ]:
# ─── GPU Detection (Windows / CUDA) ───────────────────────────────────────────
# PyTorch picks up NVIDIA GPUs automatically via CUDA.
# Make sure you have the CUDA-enabled torch installed:
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
# (change cu118 to your CUDA version, e.g. cu121 for CUDA 12.1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU             : {gpu_name}")
    print(f"VRAM            : {vram_gb:.2f} GB")
    USE_FP16 = True
    TRAIN_BATCH = 8   # bump up if you have ≥6 GB VRAM; lower to 4 if OOM
else:
    print("⚠️  No GPU found – running on CPU (training will be very slow).")
    print("    Install CUDA-enabled PyTorch to use your NVIDIA GPU.")
    USE_FP16 = False
    TRAIN_BATCH = 4

print(f"\nTraining batch size : {TRAIN_BATCH}")
print(f"FP16 mixed-precision: {USE_FP16}")


## 📊 Step 2: Data Loading & Inspection


In [ ]:
# Scan all images in data_subset
IMG_DIR = Path("data_subset")
all_images = sorted(IMG_DIR.glob("*.png"))
print(f"Total images found: {len(all_images)}")
print(f"First 5: {[p.name for p in all_images[:5]]}")
print(f"Last 5:  {[p.name for p in all_images[-5:]]}")


In [ ]:
# Display random sample images
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
sample_imgs = random.sample(all_images, 6)

for ax, img_path in zip(axes.flatten(), sample_imgs):
    img = Image.open(img_path)
    ax.imshow(img, cmap='gray')
    ax.set_title(img_path.stem, fontsize=8)
    ax.axis('off')

plt.suptitle('Random Sample Images from Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Image size statistics
widths, heights, sizes_kb = [], [], []
for p in tqdm(all_images[:500], desc="Scanning image stats"):
    img = Image.open(p)
    w, h = img.size
    widths.append(w)
    heights.append(h)
    sizes_kb.append(p.stat().st_size / 1024)

print(f"Width  - min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height - min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")
print(f"File size (KB) - min: {min(sizes_kb):.1f}, max: {max(sizes_kb):.1f}, mean: {np.mean(sizes_kb):.1f}")


In [ ]:
from pathlib import Path
import pandas as pd

# Path to file
forms_file = Path("forms_for_parsing.txt")

forms_data = []

# DEBUG: check first few lines
print("---- SAMPLE LINES FROM FILE ----")
with open(forms_file, 'r') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i == 5:
            break

print("\n---- PARSING START ----")

# Actual parsing
with open(forms_file, 'r') as f:
    for line in f:
        line = line.strip()

        # Skip empty/comment lines
        if not line or line.startswith('#'):
            continue

        parts = line.split()

        # Minimum required fields
        if len(parts) < 4:
            continue

        try:
            forms_data.append({
                'form_id': parts[0],
                'writer_id': parts[1],
                'num_sentences': int(parts[2]),
                'seg_status': parts[3]
            })
        except Exception as e:
            print(f"Skipping line due to error: {line}")
            print(e)

# Convert to DataFrame
forms_df = pd.DataFrame(forms_data)

print("\n---- RESULTS ----")

# Safety check
if forms_df.empty:
    print("❌ ERROR: No data parsed. Check file format.")
else:
    print(f"✅ Forms parsed: {len(forms_df)}")
    print(f"✅ Unique writers: {forms_df['writer_id'].nunique()}")

    print("\nSegmentation status distribution:")
    print(forms_df['seg_status'].value_counts())


## 🧹 Step 3: Data Cleaning


In [ ]:
# Remove corrupted/unreadable images
valid_images = []
corrupted = 0

for p in tqdm(all_images, desc="Validating images"):
    try:
        img = Image.open(p)
        img.verify()  # verify image integrity
        valid_images.append(str(p))
    except Exception:
        corrupted += 1

print(f"Valid images: {len(valid_images)}")
print(f"Corrupted/removed: {corrupted}")


## 🧠 Step 5: Load TrOCR Processor & Model


In [ ]:
# Load processor
MODEL_NAME = "microsoft/trocr-base-handwritten"
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
print(f"Processor loaded: {MODEL_NAME}")


In [ ]:
# Load model and move to GPU
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
model = model.to(device)

# Configure for generation
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
# model.config.max_length = 64
# model.config.num_beams = 4

total_params = sum(p.numel() for p in model.parameters())
print(f"Model on: {next(model.parameters()).device}")
print(f"Parameters: {total_params:,}")


## 🏷️ Step 5b: Generate Pseudo-Labels

Since we only have images (no ground-truth text labels in the dataset files), we use the **pre-trained TrOCR model** to generate text predictions as pseudo-labels. We then fine-tune on a subset to improve accuracy.


In [ ]:
from transformers import GenerationConfig
from PIL import Image
import torch

def predict_batch(image_paths, model, processor, device, batch_size=4):
    results = []

    # Reset any bad config
    model.config.max_length = None
    model.config.num_beams = None

    # Proper generation config
    gen_config = GenerationConfig(
        max_length=64,
        num_beams=4
    )

    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]

        images = [Image.open(p).convert("RGB") for p in batch_paths]
        pixel_values = processor(images, return_tensors="pt").pixel_values.to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                pixel_values,
                generation_config=gen_config
            )

        preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
        results.extend(preds)

    return results


In [ ]:
#load pseudo-labels from file (if already generated)
import json

with open("pseudo_labels.json", "r", encoding="utf-8") as f:
    data = json.load(f)

valid_images = data["images"]
all_labels = data["labels"]

print("Pseudo labels loaded!")


In [ ]:
# Generate pseudo-labels for all valid images
# Process in chunks to show progress
BATCH_SIZE = 4 if not torch.cuda.is_available() else 8  # larger batch on GPU
all_labels = []

for i in tqdm(range(0, len(valid_images), BATCH_SIZE), desc="Generating pseudo-labels"):
    chunk = valid_images[i:i+BATCH_SIZE]
    labels = predict_batch(chunk, model, processor, device, batch_size=BATCH_SIZE)
    all_labels.extend(labels)

print(f"\nGenerated labels for {len(all_labels)} images")
print(f"Sample predictions:")
for i in range(min(5, len(all_labels))):
    print(f"  {Path(valid_images[i]).name}: '{all_labels[i]}'")


In [ ]:
# ─── OPTIONAL: Load pseudo-labels from file (if already generated) ────────────
# If you already ran the cell above and saved labels, just load them here
# to skip the slow generation step.
#
# Uncomment to load:
# import json
# with open("pseudo_labels.json", "r", encoding="utf-8") as f:
#     data = json.load(f)
# valid_images = data["images"]
# all_labels   = data["labels"]
# print("Pseudo labels loaded!")
#
# Uncomment to save (run once after generation):
# import json
# with open("pseudo_labels.json", "w", encoding="utf-8") as f:
#     json.dump({"images": valid_images, "labels": all_labels}, f)
# print("Pseudo labels saved!")


In [ ]:
# Build dataset DataFrame
df = pd.DataFrame({
    'image_path': valid_images[:len(all_labels)],
    'label': all_labels
})

# Clean labels
df['label'] = df['label'].str.strip()
df = df[df['label'].str.len() > 0]  # remove empty
df['label_len'] = df['label'].str.len()

print(f"Dataset size: {len(df)}")
print(f"Average label length: {df['label_len'].mean():.1f} chars")
print(f"Min/Max label length: {df['label_len'].min()}/{df['label_len'].max()}")


In [ ]:
# Label length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram of label lengths
axes[0].hist(df['label_len'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Label Length (chars)')
axes[0].set_ylabel('Count')
axes[0].set_title('Label Length Distribution')
axes[0].axvline(df['label_len'].mean(), color='red', linestyle='--', label=f"Mean: {df['label_len'].mean():.1f}")
axes[0].legend()

# Word count distribution
df['word_count'] = df['label'].str.split().str.len()
axes[1].hist(df['word_count'], bins=20, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Count')
axes[1].set_title('Words per Label Distribution')

plt.tight_layout()
plt.show()

print(f"\nTop 10 label lengths:")
print(df['label_len'].value_counts().sort_index().head(10))


## 🧪 Step 4: Data Augmentation


In [ ]:
# Augmentation transforms for training
train_augment = T.Compose([
    T.RandomRotation(degrees=5),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomAffine(degrees=0, translate=(0.03, 0.03)),
])

print("Augmentation pipeline:")
print("  ✓ RandomRotation (±5°)")
print("  ✓ GaussianBlur")
print("  ✓ ColorJitter (brightness/contrast)")
print("  ✓ RandomAffine (slight translation)")


In [ ]:
# Visualize augmentation on a sample
sample_img = Image.open(df.iloc[0]['image_path']).convert('RGB')

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
axes[0].imshow(sample_img)
axes[0].set_title('Original')
axes[0].axis('off')

for i in range(1, 4):
    aug_img = train_augment(sample_img)
    axes[i].imshow(aug_img)
    axes[i].set_title(f'Augmented {i}')
    axes[i].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 🔄 Step 6: Preprocessing & Dataset Class


In [ ]:
class HTRDataset(Dataset):
    """Custom dataset for handwritten text recognition."""
    
    def __init__(self, dataframe, processor, max_len=32, augment=None):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
        self.max_len = max_len
        self.augment = augment
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load and convert image
        img = Image.open(row['image_path']).convert('RGB')
        
        # Apply augmentation
        if self.augment and random.random() > 0.5:
            img = self.augment(img)
        
        # Process image -> pixel_values
        pixel_values = self.processor(img, return_tensors='pt').pixel_values.squeeze(0)
        
        # Tokenize label
        labels = self.processor.tokenizer(
            row['label'],
            padding='max_length',
            max_length=self.max_len,
            truncation=True,
            return_tensors='pt'
        ).input_ids.squeeze(0)
        
        # Replace padding tokens with -100 for loss calculation
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        
        return {
            'pixel_values': pixel_values,
            'labels': labels
        }

print("HTRDataset class defined")


## ✂️ Step 7: Train/Validation Split


In [ ]:
# Shuffle and split 90/10
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

split_idx = int(0.9 * len(df_shuffled))
train_df = df_shuffled[:split_idx]
val_df = df_shuffled[split_idx:]

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")


In [ ]:
# Create dataset objects
MAX_SEQ_LEN = 32

train_dataset = HTRDataset(train_df, processor, max_len=MAX_SEQ_LEN, augment=train_augment)
val_dataset = HTRDataset(val_df, processor, max_len=MAX_SEQ_LEN, augment=None)

# Quick sanity check
sample = train_dataset[0]
print(f"pixel_values shape: {sample['pixel_values'].shape}")
print(f"labels shape: {sample['labels'].shape}")
print(f"Sample label (decoded): {processor.tokenizer.decode(sample['labels'][sample['labels'] != -100])}")


## 🔧 Step 8: Training Configuration

Key changes vs. original:
- **`num_train_epochs=6`** — more passes for better convergence
- **`save_strategy="steps"` + `save_steps=500`** — checkpoint every 500 steps
- **`save_total_limit=3`** — keep last 3 checkpoints to save disk space
- **`load_best_model_at_end=True`** — final model = best checkpoint by CER
- **`fp16=USE_FP16`** — mixed precision when GPU is available
- **`lr_scheduler_type="cosine"`** — smoother LR decay
- **`warmup_ratio=0.05`** — 5% warm-up of total steps


In [ ]:
# ─── Training arguments ────────────────────────────────────────────────────────
# Checkpoints are saved to ./trocr_checkpoints every 500 steps.
# You can close VS Code, reopen later, and:
#   • Resume training  : trainer.train(resume_from_checkpoint=True)
#   • Skip to testing  : load the best checkpoint directly (Step 8b below)

CHECKPOINT_DIR = "./trocr_checkpoints"

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    predict_with_generate=True,

    # ── Epochs / steps ───────────────────────────────────────────────────────
    num_train_epochs=6,

    # ── Checkpointing ────────────────────────────────────────────────────────
    save_strategy="steps",
    save_steps=500,               # save checkpoint every 500 steps
    save_total_limit=3,           # keep only last 3 checkpoints
    load_best_model_at_end=True,  # restore best checkpoint at end of training

    # ── Evaluation ───────────────────────────────────────────────────────────
    eval_strategy="steps",
    eval_steps=500,
    metric_for_best_model="cer",
    greater_is_better=False,

    # ── Batch / learning rate ─────────────────────────────────────────────────
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=TRAIN_BATCH,
    learning_rate=3e-5,           # slightly lower LR for more epochs
    warmup_ratio=0.05,            # 5% of total steps for warm-up
    weight_decay=0.01,
    lr_scheduler_type="cosine",   # cosine annealing for smoother convergence
    gradient_accumulation_steps=2,  # accumulate gradients to effectively double batch size
    dataloader_pin_memory=True,  # speed up data transfer to GPU

    # ── Mixed precision ───────────────────────────────────────────────────────
    fp16=USE_FP16,             # enabled only when CUDA is available

    # ── Logging / misc ────────────────────────────────────────────────────────
    logging_steps=100,
    logging_dir="./logs",
    dataloader_num_workers=0,     # 0 is safest on Windows
    report_to="none",
)
model.gradient_checkpointing_enable()
print("Training arguments configured")
print(f"  Epochs         : {training_args.num_train_epochs}")
print(f"  Batch size     : {training_args.per_device_train_batch_size}")
print(f"  Learning rate  : {training_args.learning_rate}")
print(f"  LR scheduler   : {training_args.lr_scheduler_type}")
print(f"  FP16           : {training_args.fp16}")
print(f"  Save every     : {training_args.save_steps} steps")
print(f"  Checkpoint dir : {CHECKPOINT_DIR}")


## 💾 Step 8b: Load Existing Checkpoint (skip training)

If you already trained and just want to run evaluation / inference,
run this cell instead of the training cell below.  
It loads the **best saved checkpoint** automatically.


In [ ]:
CHECKPOINT_DIR = "./trocr_checkpoints"


In [ ]:
# ─── Load the best checkpoint without re-training ─────────────────────────────
# Finds the checkpoint with the lowest CER saved inside CHECKPOINT_DIR.

import os
from pathlib import Path as _Path

def find_best_checkpoint(checkpoint_dir):
    """Return path to the best (lowest CER) checkpoint, or None."""
    ckpt_root = _Path(checkpoint_dir)
    if not ckpt_root.exists():
        return None
    # trainer_state.json inside each checkpoint contains eval metrics
    best_ckpt = None
    best_cer  = float('inf')
    for ckpt in sorted(ckpt_root.glob("checkpoint-*")):
        state_file = ckpt / "trainer_state.json"
        if state_file.exists():
            with open(state_file) as f:
                state = json.load(f)
            # best_metric is stored in trainer_state.json
            bm = state.get("best_metric", None)
            if bm is not None and bm < best_cer:
                best_cer  = bm
                best_ckpt = str(ckpt)
    return best_ckpt

# --- Check whether a checkpoint exists ---
best_ckpt_path = find_best_checkpoint(CHECKPOINT_DIR)

if best_ckpt_path:
    print(f"✅ Found checkpoint: {best_ckpt_path}")
    print("Loading model weights...")
    model = VisionEncoderDecoderModel.from_pretrained(best_ckpt_path).to(device)

    # Re-apply generation config after loading
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id           = processor.tokenizer.pad_token_id
    model.config.vocab_size             = model.config.decoder.vocab_size
    model.config.eos_token_id           = processor.tokenizer.sep_token_id

    print(f"✅ Model loaded from checkpoint  →  device: {next(model.parameters()).device}")
    print("   You can now jump straight to Step 9b (Evaluation) or Step 12 (Testing).")
else:
    print("ℹ️  No checkpoint found. Run the training cell below (Step 8c) first.")


## 📊 Step 9: Evaluation Metrics


In [ ]:
from jiwer import cer as compute_cer_jiwer
from jiwer import wer as compute_wer_jiwer

def compute_metrics(pred):
    """Compute CER, WER, and Exact Match for evaluation."""
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    
    # Replace -100 with pad token for decoding
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    
    # Decode
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    
    # Filter empty strings
    pairs = [(p, l) for p, l in zip(pred_str, label_str) if len(l.strip()) > 0]
    if not pairs:
        return {'cer': 1.0, 'wer': 1.0, 'exact_match': 0.0}
    
    pred_filtered, label_filtered = zip(*pairs)
    pred_filtered = list(pred_filtered)
    label_filtered = list(label_filtered)
    
    # CER
    cer_score = compute_cer_jiwer(label_filtered, pred_filtered)
    
    # WER
    wer_score = compute_wer_jiwer(label_filtered, pred_filtered)
    
    # Exact Match
    exact = sum(1 for p, l in zip(pred_filtered, label_filtered) if p.strip() == l.strip())
    exact_match = exact / len(pred_filtered)
    
    return {
        'cer': cer_score,
        'wer': wer_score,
        'exact_match': exact_match
    }

print("Metrics function defined (CER, WER, Exact Match)")


In [ ]:
# Initialize trainer
trainer = Seq2SeqTrainer(
    model=model,
    processing_class=processor.image_processor,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=default_data_collator,
)

print("Seq2SeqTrainer initialized")


## 🚀 Step 8c: Train

- Checkpoints are saved to `./trocr_checkpoints/checkpoint-<step>` every 500 steps.  
- If VS Code / kernel crashes, resume with `trainer.train(resume_from_checkpoint=True)`.  
- To skip training entirely, run **Step 8b** above to load the best saved checkpoint.


In [ ]:
# ─── Train! ───────────────────────────────────────────────────────────────────
# To resume from the latest checkpoint (after a crash / restart), change
# resume_from_checkpoint=True

print("Starting training...")
train_result = trainer.train(resume_from_checkpoint=False)

# Save final model
model.save_pretrained("trained_trocr")
processor.save_pretrained("trained_trocr")

print(f"\n{'='*50}")
print("TRAINING COMPLETE")
print(f"{'='*50}")
print(f"Training loss:  {train_result.training_loss:.4f}")
print(f"Training time:  {train_result.metrics.get('train_runtime', 0):.1f}s")
print(f"Model saved to: ./trained_trocr")


In [ ]:
import json

state_path = "trocr_checkpoints/checkpoint-1500/trainer_state.json"

with open(state_path) as f:
    state = json.load(f)

log_history = state["log_history"]


In [ ]:
# Plot training loss curve — loads history from saved checkpoint file
import json
from pathlib import Path

# Load log_history from the best checkpoint's trainer_state.json
ckpt_root = Path(CHECKPOINT_DIR)
all_states = []
for ckpt in sorted(ckpt_root.glob("checkpoint-*")):
    state_file = ckpt / "trainer_state.json"
    if state_file.exists():
        with open(state_file) as f:
            state = json.load(f)
        all_states.append((ckpt.name, state))

if not all_states:
    print("No trainer_state.json found in any checkpoint folder.")
else:
    # Use the latest checkpoint's state (it has the full history)
    ckpt_name, state = all_states[-1]
    log_history = state["log_history"]
    print(f"Loaded log_history from: {ckpt_name}  ({len(log_history)} entries)")

    train_steps = [x['step'] for x in log_history if 'loss' in x]
    train_loss  = [x['loss'] for x in log_history if 'loss' in x]

    eval_steps = [x['step'] for x in log_history if 'eval_loss' in x]
    eval_loss  = [x['eval_loss'] for x in log_history if 'eval_loss' in x]
    eval_cer   = [x['eval_cer'] for x in log_history if 'eval_cer' in x]
    eval_em    = [x['eval_exact_match'] for x in log_history if 'eval_exact_match' in x]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].plot(train_steps, train_loss, 'b-', alpha=0.7, label='Training Loss')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    if eval_loss:
        axes[1].plot(eval_steps, eval_loss, 'r-o', label='Validation Loss')
        axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss')
        axes[1].set_title('Validation Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    if eval_cer:
        axes[2].plot(eval_steps, eval_cer, 'g-o', label='CER')
        if eval_em:
            ax2 = axes[2].twinx()
            ax2.plot(eval_steps, eval_em, 'm--s', label='Exact Match')
            ax2.set_ylabel('Exact Match')
            ax2.legend(loc='lower right')
        axes[2].set_xlabel('Step'); axes[2].set_ylabel('CER')
        axes[2].set_title('CER & Exact Match'); axes[2].legend(loc='upper right'); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    if train_loss: print(f"Final training loss: {train_loss[-1]:.4f}")
    if eval_loss:  print(f"Best eval loss:      {min(eval_loss):.4f}")
    if eval_cer:   print(f"Best CER:            {min(eval_cer):.4f}")
    if eval_em:    print(f"Best Exact Match:    {max(eval_em):.4f}")


## 📊 Step 9b: Full Evaluation


In [ ]:
# Run full evaluation on validation set — no trainer.train() required
from jiwer import cer as compute_cer_jiwer, wer as compute_wer_jiwer
from transformers import GenerationConfig

# Clear any generation params from model.config
model.config.max_length = None
model.config.num_beams = None

model.eval()
all_preds, all_labels_list = [], []

eval_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=16, shuffle=False,
    collate_fn=default_data_collator
)

# Include decoder_start_token_id to fix ValueError
gen_config = GenerationConfig(
    max_new_tokens=64,
    num_beams=1,  # greedy — much faster, nearly same accuracy
    decoder_start_token_id=processor.tokenizer.cls_token_id,
)

print("Running evaluation on validation set...")
for batch in tqdm(eval_loader, desc="Evaluating"):
    pixel_values = batch["pixel_values"].to(device)
    labels = batch["labels"].clone()

    with torch.no_grad():
        generated_ids = model.generate(pixel_values, generation_config=gen_config)

    preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
    all_preds.extend(preds)

    labels[labels == -100] = processor.tokenizer.pad_token_id
    labs = processor.batch_decode(labels, skip_special_tokens=True)
    all_labels_list.extend(labs)

# Filter empty labels
pairs = [(p, l) for p, l in zip(all_preds, all_labels_list) if len(l.strip()) > 0]
pred_filtered, label_filtered = zip(*pairs)

eval_cer_score   = compute_cer_jiwer(list(label_filtered), list(pred_filtered))
eval_wer_score   = compute_wer_jiwer(list(label_filtered), list(pred_filtered))
exact_count      = sum(p.strip() == l.strip() for p, l in zip(pred_filtered, label_filtered))
eval_exact_score = exact_count / len(pred_filtered)

print(f"{'='*50}")
print("EVALUATION RESULTS")
print(f"{'='*50}")
print(f"CER:         {eval_cer_score:.4f}")
print(f"WER:         {eval_wer_score:.4f}")
print(f"Exact Match: {eval_exact_score:.4f}")


In [ ]:
# Show sample predictions vs ground truth
model.eval()
n_samples = 10
sample_indices = random.sample(range(len(val_df)), min(n_samples, len(val_df)))

print(f"{'='*70}")
print("SAMPLE PREDICTIONS vs GROUND TRUTH")
print(f"{'='*70}")

correct = 0
for i, idx in enumerate(sample_indices):
    row = val_df.iloc[idx]
    img = Image.open(row['image_path']).convert('RGB')
    
    pixel_values = processor(img, return_tensors='pt').pixel_values.to(device)
    with torch.no_grad():
        gen_ids = model.generate(pixel_values, max_length=64, num_beams=4)
    pred = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    
    gt = row['label']
    match = pred.strip() == gt.strip()
    if match:
        correct += 1
    
    status = '✅' if match else '❌'
    print(f"\n{status} Sample {i+1}:")
    print(f"   Ground Truth: {gt}")
    print(f"   Prediction:   {pred}")

print(f"\n{'='*70}")
print(f"Accuracy on samples: {correct}/{len(sample_indices)} = {correct/len(sample_indices)*100:.1f}%")


## 🔥 Step 10: Performance Boosting - Add Custom Names


In [ ]:
# Create synthetic name entries to mix with dataset
custom_names = [
    "gaurav", "rahul", "amit", "priya", "sneha",
    "arjun", "deepak", "neha", "ravi", "anand",
    "john", "james", "michael", "david", "robert",
    "william", "richard", "joseph", "thomas", "daniel",
]

print(f"Custom name dictionary: {len(custom_names)} names")
print(f"Names: {', '.join(custom_names)}")
print("\nThese will be used for spell correction in Step 13")


## 🔮 Step 11: Inference Function


In [ ]:
def predict_text(image_path, model=model, processor=processor, device=device):
    """Predict text from a single handwritten image."""
    # Load image
    img = Image.open(image_path).convert('RGB')
    
    # Preprocess
    pixel_values = processor(img, return_tensors='pt').pixel_values.to(device)
    
    # Generate
    model.eval()
    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=64,
            num_beams=4
        )
    
    # Decode
    predicted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return predicted_text

print("predict_text() function ready")


## 🧪 Step 12: Testing


In [ ]:
# Test on 10 random images with visualization
test_images = random.sample(valid_images, min(10, len(valid_images)))

fig, axes = plt.subplots(5, 2, figsize=(16, 15))
axes = axes.flatten()

print(f"{'='*60}")
print("TEST PREDICTIONS")
print(f"{'='*60}")

for i, img_path in enumerate(test_images):
    pred = predict_text(img_path)
    
    # Show image
    img = Image.open(img_path)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Pred: {pred}", fontsize=9, color='darkblue')
    axes[i].axis('off')
    
    print(f"  Image: {Path(img_path).name}")
    print(f"  Prediction: {pred}")
    print()

plt.suptitle('Test Predictions on Random Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## ⚡ Bonus: Spell Correction for Short Words


In [ ]:
from difflib import get_close_matches

# Extended dictionary of common names
name_dictionary = {
    'gaurav', 'rahul', 'amit', 'priya', 'sneha', 'arjun', 'deepak',
    'neha', 'ravi', 'anand', 'aman', 'abhishek', 'aryan', 'arun',
    'john', 'james', 'michael', 'david', 'robert', 'william',
    'richard', 'joseph', 'charles', 'thomas', 'christopher',
    'daniel', 'matthew', 'anthony', 'mark', 'donald', 'steven',
    'kumar', 'divya', 'ashok', 'vijay', 'suresh', 'rajesh',
    'move', 'stop', 'from', 'the', 'and', 'for', 'was', 'are',
    'but', 'not', 'you', 'all', 'can', 'had', 'her', 'one',
    'our', 'out', 'has', 'his', 'how', 'man', 'new', 'now',
}

def spell_correct(text, dictionary=name_dictionary, threshold=0.75):
    """Correct short words using closest dictionary match."""
    words = text.split()
    corrected = []
    for word in words:
        if len(word) <= 10:  # only correct short words
            matches = get_close_matches(word.lower(), dictionary, n=1, cutoff=threshold)
            if matches:
                corrected.append(matches[0])
            else:
                corrected.append(word)
        else:
            corrected.append(word)
    return ' '.join(corrected)

print(f"Spell correction dictionary: {len(name_dictionary)} words")


In [ ]:
# Test spell correction on predictions
test_sample = random.sample(valid_images, min(8, len(valid_images)))

print(f"{'='*70}")
print("PREDICTIONS WITH SPELL CORRECTION")
print(f"{'='*70}")

for i, img_path in enumerate(test_sample):
    raw_pred = predict_text(img_path)
    corrected = spell_correct(raw_pred)
    
    changed = '🔄' if raw_pred != corrected else '✓'
    print(f"\n{changed} Sample {i+1} ({Path(img_path).name}):")
    print(f"   Raw Prediction: {raw_pred}")
    print(f"   Corrected:      {corrected}")

print(f"\n{'='*70}")
print("HTR System Ready! ✅")


In [ ]:
def predict(image_path):
    img = Image.open(image_path).convert("RGB")

    pixel_values = processor(
        images=img,
        return_tensors="pt"
    ).pixel_values.to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=50,
            num_beams=4
        )

    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]


In [ ]:
# Example inference
# Replace the path with your own handwritten line image.
# Example: print(predict("path/to/handwritten_image.png"))

